In [ ]:
# NCrystal notebook settings
# title: NCrystal materials in McStas
# shortkey: nw12mcstas
# section: neuwave12
# requires: plot, mcstas
# max-line-length: 174

In this example we install McStas and Ncrystal using conda / mamba, and create a instrument simulation for a simple transmission beamline with a NCrystal sample.

### Additional modules:

In [ ]:
import os
import pathlib
import shutil
import numpy as np
import shlex
import matplotlib.pyplot as plt

## Example

This cell writes the McStas instrument file. We write it this way because in Google Colab files are deleted when the runtime is regenerated.

We also define a few variables that will be used in the simulation.

In [ ]:
beam_width = 0.01 # m
beam_height = 0.01 # m
sample_thickness = 0.01 # m
nbins = 700
sample_cfgstr = 'C_sg194_pyrolytic_graphite.ncmat;temp=20C'
NParticles = int(1e6)#number of neutrons to simulate
source_mon1_distance = 0.5 # m
mon1_sample_distance = 0.5 # m
sample_mon2_distance = 0.5 # m
min_lambda = 1.0
max_lambda = 8.0

# The following string defines an example instrument file (based on
# NCrystal_example.comp from the McStas distribution) in which
# neutrons are directed through a monitor, then through an NCrystal_sample,
# and finally again pass a second monitor.
# 
# The sample_cfg (string) parameter accepts an NCrystal cfg-string, but
# other parameters are simply hard-coded in the text below (which is good
# enough for the purposes of the present notebook).

txt = f'''

DEFINE INSTRUMENT NCrystal_transm_example(string sample_cfg="Al_sg225.ncmat;temp=300K")

DECLARE
%{{
%}}

INITIALIZE
%{{
%}}

TRACE

COMPONENT origin = Progress_bar()
  AT (0, 0, 0) RELATIVE ABSOLUTE

 COMPONENT source =   Source_gen(Lmin={min_lambda}, Lmax={max_lambda}, xwidth={beam_width}, yheight={beam_height}, focus_xw={beam_width}, focus_yh={beam_height}, dist=100)
   AT (0, 0, 0) RELATIVE origin

COMPONENT monitor1 = L_monitor(xmin={-beam_width/2}, xmax={beam_width/2}, ymin={-beam_height/2}, ymax={beam_height/2}, nL={nbins}, filename="monitor1.dat", Lmin={min_lambda},
                     Lmax={max_lambda},restore_neutron=1)
  AT (0, 0, {source_mon1_distance}) RELATIVE source

SPLIT COMPONENT powder_sample = NCrystal_sample(xwidth=10.0, yheight=10.0,zdepth={sample_thickness},
                                                cfg=sample_cfg)
 AT (0, 0, {mon1_sample_distance}) RELATIVE monitor1

COMPONENT monitor2 = L_monitor(xmin={-beam_width/2}, xmax={beam_width/2}, ymin={-beam_height/2}, ymax={beam_height/2}, nL={nbins}, filename="monitor2.dat", Lmin={min_lambda},
                     Lmax={max_lambda},restore_neutron=1)
  AT (0, 0, {sample_mon2_distance}) RELATIVE powder_sample

END
'''

instrument_file = pathlib.Path('./transmission_example.instr').absolute()
if not instrument_file.is_file() or txt!=instrument_file.read_text():
    instrument_file.write_text(txt)

Check the instrument file:

In [ ]:
!cat {instrument_file}

Now we run McStas:

In [ ]:
outdir = pathlib.Path('./rundir')
shutil.rmtree(outdir,ignore_errors=True)
cfgstr_for_mcrun=shlex.quote(sample_cfgstr.replace(';',r'\;'))#quote for shell + workaround mcrun issues with semicolons.
!mcrun -c {instrument_file} sample_cfg={cfgstr_for_mcrun} -d {outdir} --ncount={NParticles}

Plot incident monitor

In [ ]:
datafile = outdir / 'monitor1.dat'
assert datafile.exists(), "Monitor file not found"
!head -30 {datafile}

x=np.genfromtxt(datafile)
L, I, I_err, N = x.T

fig=plt.figure()
plt.errorbar(L,I,I_err)
plt.xlabel('wavelength (Å)')
plt.xlim(min_lambda, max_lambda)
plt.show()

Plot transmission monitor

In [ ]:
datafile = outdir / 'monitor2.dat'
assert datafile.exists(), "Monitor file not found"
!head -30 {datafile}

x=np.genfromtxt(datafile)
L2, I2, I2_err, N = x.T

fig=plt.figure()
plt.errorbar(L2,I2,I2_err)
plt.xlabel('wavelength (Å)')
plt.xlim(min_lambda, max_lambda)
plt.show()

Plot transmission and compare with the theoretical (uncollided) transmission:

In [ ]:
wl = np.linspace(min_lambda, max_lambda, 10000)
mat1 = NC.load(sample_cfgstr)
xs1 = mat1.scatter.xsect(wl=wl)+mat1.absorption.xsect(wl=wl)

N1 = mat1.info.getNumberDensity()

mat1.dump()

In [ ]:
fig=plt.figure()
sample_thickness_cm = sample_thickness*100
plt.plot(wl, np.exp(-N1*xs1*sample_thickness_cm),
         color='b',
         label='Theoretical transmission')
plt.errorbar( L2, I2/I, I2_err/I, 
             color='r',ecolor=(1,0,0,0.2),fmt='.',markersize=1,
             label='McStas simulation results')
plt.xlabel('wavelength (Å)')
plt.xlim(min_lambda, max_lambda)
plt.legend()
plt.show()

## Exercise

- Reduce the sample-detector distance to study the effect of in-scattering.
- Modify the material to see effects of changing temperature or density.

In [ ]:
## Your code